In [3]:
#!/usr/bin/env python3
"""
BarentsWatch Arctic Vessel Tracker
Fetches all vessels and filters for Arctic region (above 65°N)
"""

import requests
import json
from datetime import datetime

# Your proven working credentials
CLIENT_ID = "henrikformoe@gmail.com:ArcticShadowTrackerAIS"
CLIENT_SECRET = "Xw5yCEXT5gMi5PJEKEW6"
SCOPE = "ais"

def get_access_token():
    """Get access token using proven credentials"""
    print("Getting access token...")
    
    token_url = "https://id.barentswatch.no/connect/token"
    
    data = {
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': SCOPE,
        'grant_type': 'client_credentials'
    }
    
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    
    try:
        response = requests.post(token_url, data=data, headers=headers)
        response.raise_for_status()
        
        token_data = response.json()
        access_token = token_data['access_token']
        
        print("✅ Got access token")
        return access_token
        
    except Exception as e:
        print(f"❌ Token request failed: {e}")
        return None

def get_arctic_vessels(access_token, min_latitude=65.0):
    """Get all vessels and filter for Arctic region (above specified latitude)"""
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Accept': 'application/json'
    }
    
    # Get all latest vessel positions
    url = "https://live.ais.barentswatch.no/v1/latest/combined"
    
    print(f"\n🌊 Fetching all vessels from BarentsWatch...")
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        all_vessels = response.json()
        
        # Filter for vessels above the minimum latitude
        arctic_vessels = [
            vessel for vessel in all_vessels 
            if vessel.get('latitude', 0) >= min_latitude
        ]
        
        print(f"✅ Found {len(arctic_vessels)} vessels above {min_latitude}°N")
        print(f"   (Out of {len(all_vessels)} total vessels)")
        
        return arctic_vessels
        
    except Exception as e:
        print(f"❌ Error fetching vessels: {e}")
        return []

def display_arctic_vessels(vessels):
    """Display Arctic vessel information"""
    
    if not vessels:
        print("\n❌ No Arctic vessels found")
        return
    
    print(f"\n🚢 Arctic Vessels (sample of first 10):")
    print("=" * 60)
    
    # Show first 10 vessels as sample
    for vessel in vessels[:10]:
        name = vessel.get('name', 'Unknown')
        mmsi = vessel.get('mmsi', 'N/A')
        lat = vessel.get('latitude', 0)
        lon = vessel.get('longitude', 0)
        ship_type = vessel.get('shipType', 'N/A')
        
        print(f"  • {name:25} MMSI:{mmsi:9} Pos:{lat:.2f}°N, {lon:.2f}°E Type:{ship_type}")
    
    if len(vessels) > 10:
        print(f"\n  ... and {len(vessels) - 10} more vessels")
    
    # Show statistics by ship type
    print(f"\n📊 Statistics by vessel type:")
    ship_types = {}
    for vessel in vessels:
        ship_type = vessel.get('shipType', 'Unknown')
        ship_types[ship_type] = ship_types.get(ship_type, 0) + 1
    
    for ship_type, count in sorted(ship_types.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  Type {ship_type}: {count} vessels")

def main():
    """Main function to get Arctic vessels"""
    print("🗺️  BarentsWatch Arctic Vessel Tracker")
    print("Fetching vessels above 65°N")
    print("=" * 60)
    
    # Step 1: Get access token
    access_token = get_access_token()
    if not access_token:
        print("❌ Failed to get access token. Exiting.")
        return
    
    # Step 2: Get Arctic vessels
    arctic_vessels = get_arctic_vessels(access_token, min_latitude=65.0)
    
    # Step 3: Display results
    display_arctic_vessels(arctic_vessels)
    
    # Optional: Save to JSON file
    if arctic_vessels:
        filename = f"arctic_vessels_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w') as f:
            json.dump(arctic_vessels, f, indent=2)
        print(f"\n💾 Data saved to {filename}")

if __name__ == "__main__":
    main()

🗺️  BarentsWatch Arctic Vessel Tracker
Fetching vessels above 65°N
Getting access token...
✅ Got access token

🌊 Fetching all vessels from BarentsWatch...
✅ Found 1513 vessels above 65.0°N
   (Out of 4177 total vessels)

🚢 Arctic Vessels (sample of first 10):
  • BERGSFJORD                MMSI:257898600 Pos:70.29°N, 22.26°E Type:65
  • KIM ROGER                 MMSI:257747800 Pos:67.89°N, 13.03°E Type:30
  • STRIL LUNA                MMSI:257597000 Pos:72.49°N, 20.30°E Type:90
  • NORNE                     MMSI:257069000 Pos:66.03°N, 8.09°E Type:99
  • STOLT ORION               MMSI:257747900 Pos:66.39°N, 12.83°E Type:70
  • PELAGIA FJORD             MMSI:259030300 Pos:68.51°N, 16.14°E Type:73
  • SVARTSKJAER               MMSI:257069160 Pos:65.68°N, 12.13°E Type:52
  • KV FARM                   MMSI:257069200 Pos:68.70°N, 15.42°E Type:55
  • TJOETTAGUTTEN             MMSI:257069260 Pos:65.68°N, 12.13°E Type:79
  • NYHEIM                    MMSI:259030490 Pos:68.93°N, 17.13°E Type:70



In [1]:
#!/usr/bin/env python3
"""
Simple BarentsWatch AIS Data Fetcher
Uses proven working credentials and known working MMSI
"""

import requests
import json
import time
from datetime import datetime

# Your proven working credentials
CLIENT_ID = "henrikformoe@gmail.com:ArcticShadowTrackerAIS"
CLIENT_SECRET = "Xw5yCEXT5gMi5PJEKEW6"
SCOPE = "ais"

# Known working vessel
KNOWN_MMSI = 257111020  # OV_HEKKINGEN - we verified this works

def get_access_token():
    """Get access token using proven credentials"""
    print("Getting access token...")
    
    token_url = "https://id.barentswatch.no/connect/token"
    
    data = {
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': SCOPE,
        'grant_type': 'client_credentials'
    }
    
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    
    try:
        response = requests.post(token_url, data=data, headers=headers)
        response.raise_for_status()
        
        token_data = response.json()
        access_token = token_data['access_token']
        expires_in = token_data.get('expires_in', 3600)
        
        print(f"✅ Got access token (expires in {expires_in} seconds)")
        return access_token
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Token request failed: {e}")
        return None
    except KeyError as e:
        print(f"❌ Unexpected token response format: {e}")
        return None

def get_vessel_tracks(access_token, mmsi, hours=24):
    """Get vessel tracking data for specific MMSI"""
    print(f"Fetching tracks for MMSI {mmsi} (last {hours} hours)...")
    
    # Using the proven working endpoint structure
    ais_url = f"https://historic.ais.barentswatch.no/v1/historic/trackslast{hours}hours/{mmsi}"
    
    headers = {
        'Authorization': f'Bearer {access_token}'
    }
    
    try:
        response = requests.get(ais_url, headers=headers)
        response.raise_for_status()
        
        track_data = response.json()
        print(f"✅ Got track data for MMSI {mmsi}")
        return track_data
        
    except requests.exceptions.RequestException as e:
        print(f"❌ AIS request failed: {e}")
        if hasattr(e.response, 'text'):
            print(f"Response: {e.response.text}")
        return None

def extract_latest_position(track_data):
    """Extract the most recent position from track data"""
    if not track_data:
        return None
        
    # Handle different possible data structures
    positions = []
    
    if isinstance(track_data, list):
        positions = track_data
    elif isinstance(track_data, dict):
        # Try common field names
        for field in ['positions', 'tracks', 'data', 'points']:
            if field in track_data:
                positions = track_data[field]
                break
    
    if not positions:
        print("⚠️  No position data found in response")
        return None
    
    # Get the latest position (assuming they're sorted by time)
    latest = positions[-1] if isinstance(positions, list) else positions
    
    return latest

def print_vessel_info(position_data, mmsi):
    """Print vessel information in a readable format"""
    if not position_data:
        print(f"❌ No data for MMSI {mmsi}")
        return
    
    print(f"\n🚢 Vessel Information (MMSI: {mmsi})")
    print("=" * 50)
    
    # Try to extract common fields (field names may vary)
    fields_to_check = {
        'name': ['name', 'vesselName', 'shipName'],
        'latitude': ['latitude', 'lat', 'y'],
        'longitude': ['longitude', 'lon', 'lng', 'x'],
        'speed': ['speed', 'sog', 'speedOverGround'],
        'course': ['course', 'cog', 'courseOverGround'],
        'timestamp': ['timestamp', 'time', 'dateTime', 'msgTime']
    }
    
    vessel_info = {}
    for field, possible_names in fields_to_check.items():
        for name in possible_names:
            if name in position_data:
                vessel_info[field] = position_data[name]
                break
    
    # Display what we found
    if 'name' in vessel_info:
        print(f"Name: {vessel_info['name']}")
    if 'latitude' in vessel_info and 'longitude' in vessel_info:
        print(f"Position: {vessel_info['latitude']:.4f}°N, {vessel_info['longitude']:.4f}°E")
    if 'speed' in vessel_info:
        print(f"Speed: {vessel_info['speed']} knots")
    if 'course' in vessel_info:
        print(f"Course: {vessel_info['course']}°")
    if 'timestamp' in vessel_info:
        print(f"Last Update: {vessel_info['timestamp']}")
    
    print("\n📋 Raw Data Structure:")
    print(json.dumps(position_data, indent=2)[:500] + "..." if len(str(position_data)) > 500 else json.dumps(position_data, indent=2))

def main():
    """Main function to test the AIS data fetching"""
    print("🛰️  BarentsWatch AIS Data Fetcher")
    print("Using proven credentials and known working MMSI")
    print(f"Testing with: {KNOWN_MMSI} (OV_HEKKINGEN)")
    print("=" * 60)
    
    # Step 1: Get access token
    access_token = get_access_token()
    if not access_token:
        print("❌ Failed to get access token. Exiting.")
        return
    
    # Step 2: Fetch vessel tracking data
    track_data = get_vessel_tracks(access_token, KNOWN_MMSI)
    if not track_data:
        print("❌ Failed to get tracking data. Exiting.")
        return
    
    # Step 3: Extract and display latest position
    latest_position = extract_latest_position(track_data)
    print_vessel_info(latest_position, KNOWN_MMSI)
    
    # Step 4: Success confirmation
    print("\n✅ SUCCESS: AIS data fetching is working!")
    print("You can now:")
    print("1. Add more known working MMSIs")
    print("2. Scale to multiple vessels") 
    print("3. Add satellite data integration")
    print("4. Build your visualization")

if __name__ == "__main__":
    main()

🛰️  BarentsWatch AIS Data Fetcher
Using proven credentials and known working MMSI
Testing with: 257111020 (OV_HEKKINGEN)
Getting access token...
✅ Got access token (expires in 3600 seconds)
Fetching tracks for MMSI 257111020 (last 24 hours)...
✅ Got track data for MMSI 257111020

🚢 Vessel Information (MMSI: 257111020)
Name: OV_HEKKINGEN
Position: 65.3722°N, 12.1935°E
Speed: 0 knots
Course: 126.6°

📋 Raw Data Structure:
{
  "courseOverGround": 126.6,
  "latitude": 65.372205,
  "longitude": 12.193482,
  "name": "OV_HEKKINGEN",
  "rateOfTurn": 0,
  "shipType": 54,
  "speedOverGround": 0,
  "trueHeading": 30,
  "navigationalStatus": 5,
  "mmsi": 257111020,
  "msgtime": "2025-09-17T18:37:28+00:00"
}

✅ SUCCESS: AIS data fetching is working!
You can now:
1. Add more known working MMSIs
2. Scale to multiple vessels
3. Add satellite data integration
4. Build your visualization
